This notebook documents the data collection workflow for SNMA Assignment 1.

In [ ]:
import sys
from googleapiclient.discovery import build
import json
from datetime import datetime, timezone
from langdetect import detect

/Users/akshitaagrawal/anaconda3/lib/python3.10/site-packages/google/api_core/_python_version_support.py:263: FutureWarning: You are using a Python version (3.10.9) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [2]:
# creating youtube client
def youtubeClient():
    try:
        # add api key here
        apiKey = "API-Key"

        youtube = build('youtube', 'v3', developerKey=apiKey)
    except Exception as e:
        sys.stderr.write("Failed to create YouTube client: {}\n".format(str(e)))
        sys.exit(1)

    return youtube

In [3]:
# filtering out non-english comments and video titles
def is_english(text, min_len=15):
    text = (text or "").strip()

    # keeping very short comments as emojis or slang may be useful for sentiment analysis
    if len(text) <= min_len:
        return True

    try:
        return detect(text) == 'en'
    except:
        return False

In [ ]:
# searching for relevant videos and filtering them based on number of comments

def getRelevantVideos(queries, maxVideosPerQuery=50, minComments=50):
    client = youtubeClient()

    print("\nSearching for videos...\n")
    
    all_videos = {}
    
    # only keep videos published after 3 May 2026 and by 9 May 2026
    start_date = datetime(2026, 5, 3, tzinfo=timezone.utc)
    end_date = datetime(2026, 5, 9, 23, 59, 59, tzinfo=timezone.utc)

    for query in queries:
        print(f"Query: {query}")

        response = client.search().list(
            q=query,
            part='snippet',
            type='video',
            maxResults=maxVideosPerQuery
        ).execute()

        for item in response.get('items', []):
            snippet = item['snippet']
            
            title_text = snippet.get('title', '')
            # keeping videos that have english titles
            if not is_english(title_text, min_len=0):
                continue
            
            videoId = item['id']['videoId']
            
            # parse publishedAt for date filtering
            published_at_str = snippet['publishedAt']  # e.g. "2026-02-03T12:34:56Z"
            published_at = datetime.fromisoformat(
                published_at_str.replace("Z", "+00:00")
            )

            # keep only videos posted after 3 May 2026 and before 10 May 2026
            if not (start_date <= published_at <= end_date):
                continue

            title = snippet['title'].lower()
            description = snippet.get('description', '').lower()

            if "met gala" in title or "met gala" in description:
                all_videos[videoId] = snippet

    print(f"\nFound {len(all_videos)} unique relevant videos")

    if not all_videos:
        return [], {}

    def chunk_list(lst, size=50):
        for i in range(0, len(lst), size):
            yield lst[i:i + size]

    video_ids = list(all_videos.keys())
    filtered_videos = {}
    stats_map = {}

    print("\nFetching comment counts...\n")

    for chunk in chunk_list(video_ids):
        response = client.videos().list(
            id=",".join(chunk),
            part="statistics"
        ).execute()

        for item in response.get('items', []):
            vid = item['id']
            stats = item['statistics']

            comment_count = int(stats.get('commentCount', 0))

            if comment_count >= minComments:
                filtered_videos[vid] = all_videos[vid]
                stats_map[vid] = comment_count

    print(f"\nVideos after filtering (≥ {minComments} comments): {len(filtered_videos)}")

    sorted_video_ids = sorted(
        filtered_videos.keys(),
        key=lambda vid: stats_map[vid],
        reverse=True
    )

    return sorted_video_ids, filtered_videos

In [ ]:
# fetching data from the relevant videos found

def fetchData(video_ids, video_snippets, maxCommentsTotal=30000, outputFile='youtubeData-MetGala.json'):
    client = youtubeClient()
    
    # only keep comments and replies published after 3 May 2026 and by 9 May 2026
    start_date = datetime(2026, 5, 3, tzinfo=timezone.utc)
    end_date = datetime(2026, 5, 9, 23, 59, 59, tzinfo=timezone.utc)

    print("\nFetching video statistics...\n")
    
    def chunk_list(lst, size=50):
        for i in range(0, len(lst), size):
            yield lst[i:i + size]

    # fetching video statistics (viewCount, likeCount)
    video_stats = {}

    for chunk in chunk_list(video_ids):
        response = client.videos().list(
            id=",".join(chunk),
            part="statistics"
        ).execute()

        for item in response.get('items', []):
            video_stats[item['id']] = item['statistics']

    print("Video stats fetched\n")
    
    print("Fetching comments...\n")

    videos = []
    total_comments = 0

    for videoId in video_ids:

        if total_comments >= maxCommentsTotal:
            break

        snippet = video_snippets[videoId]
        stats = video_stats.get(videoId, {})

        video = {
            'title': snippet['title'],
            'videoId': videoId,
            'channelId': snippet['channelId'],
            'channelTitle': snippet['channelTitle'],
            'publishedAt': snippet['publishedAt'],
            'viewCount': int(stats.get('viewCount', 0)),
            'likeCount': int(stats.get('likeCount', 0)),
            'comments': []
        }

        comments_fetched = 0
        next_page_token = None

        try:
            while True:
                response = client.commentThreads().list(
                    videoId=videoId,
                    part='snippet,replies',
                    maxResults=100,
                    pageToken=next_page_token,
                    textFormat='plainText'
                ).execute()

                for item in response.get('items', []):
                    comment = item['snippet']['topLevelComment']['snippet']
                    
                    comment_text = comment.get('textDisplay', '')
                    # keeping only english comments
                    if not is_english(comment_text):
                        continue
                    
                    # filtering comments by date
                    comment_date = datetime.fromisoformat(
                        comment['publishedAt'].replace("Z", "+00:00")
                    )

                    if not (start_date <= comment_date <= end_date):
                        continue

                    top_comment = item['snippet']['topLevelComment']
                    
                    # adding parent comment data
                    video['comments'].append({
                        'commentId': top_comment['id'],
                        'authorId': comment.get('authorChannelId', {}).get('value'),
                        'author': comment.get('authorDisplayName'),
                        'text': comment.get('textDisplay'),
                        'publishedAt': comment.get('publishedAt'),
                        'likeCount': comment.get('likeCount', 0),
                        'totalReplyCount': item['snippet'].get('totalReplyCount', 0),
                        'parentCommentId': None,
                        'replyToAuthorId': None,
                        'videoId': videoId,
                        'isReply': False
                    })

                    # adding additional data if comment has replies
                    if 'replies' in item:
                        parent_comment_id = top_comment['id']
                        parent_author_id = (comment.get('authorChannelId', {}).get('value'))

                        for reply in item['replies']['comments']:
                            reply_snippet = reply['snippet']
                            
                            reply_text = reply_snippet.get('textDisplay', '')
                            # keeping replies that are in english only
                            if not is_english(reply_text):
                                continue
                            
                            # filtering replies by date
                            reply_date = datetime.fromisoformat(
                                reply_snippet['publishedAt'].replace("Z", "+00:00")
                            )

                            if not (start_date <= reply_date <= end_date):
                                continue

                            video['comments'].append({
                                'commentId': reply['id'],
                                'authorId': reply_snippet.get('authorChannelId', {}).get('value'),
                                'author': reply_snippet.get('authorDisplayName'),
                                'text': reply_snippet.get('textDisplay'),
                                'publishedAt': reply_snippet.get('publishedAt'),
                                'likeCount': reply_snippet.get('likeCount', 0),
                                'parentCommentId': parent_comment_id,
                                'replyToAuthorId': parent_author_id,
                                'videoId': videoId,
                                'isReply': True
                            })

                    total_comments += 1
                    comments_fetched += 1

                next_page_token = response.get('nextPageToken')

                if not next_page_token:
                    break

            print(f"{snippet['title'][:60]}... → {comments_fetched} comments")

        except Exception as e:
            print(f"{snippet['title'][:50]}... → Error: {e}")

        videos.append(video)

    with open(outputFile, 'w', encoding='utf-8') as f:
        json.dump({'videos': videos}, f, indent=2, ensure_ascii=False)

    print("\n==============================")
    print(f"Videos collected: {len(videos)}")
    print(f"Total comments collected: {total_comments}")
    print(f"Saved to: {outputFile}")
    print("==============================\n")

In [6]:
queries = [
    '"Met Gala 2026"',
    '"Met Gala 2026" live',
    '"Met Gala 2026" reaction',
    '"Met Gala 2026" review',
    '"Met Gala 2026" looks',
    '"Met Gala 2026" theme'
]

In [7]:
video_ids, video_snippets = getRelevantVideos(
    queries,
    maxVideosPerQuery=50,
    minComments=50
)


Searching for videos...

Query: "Met Gala 2026"
Query: "Met Gala 2026" live
Query: "Met Gala 2026" reaction
Query: "Met Gala 2026" review
Query: "Met Gala 2026" looks
Query: "Met Gala 2026" theme

Found 99 unique relevant videos

Fetching comment counts...


Videos after filtering (≥ 50 comments): 70


In [8]:
fetchData(
    video_ids,
    video_snippets,
    maxCommentsTotal=30000,
    outputFile='youtubeData-MetGala.json'
)


Fetching video statistics...

Video stats fetched

Fetching comments...

Live at Met Gala 2026 With Vogue... → 4200 comments
2026 MET GALA FASHION ROAST (just cancel the whole thing)... → 2408 comments
A WILD RACHEL ZEGLER APPEARS! – The Annual Met Gala 2026 REV... → 2166 comments
Beyonce &amp; Rihanna OUTSHINE the Met Gala | Olandria SNUBB... → 1330 comments
Best Dressed at the Met Gala 2026... → 889 comments
Heidi Klum Turns to Stone for the 2026 Met Gala... → 976 comments
Met Gala 2026 Outfit Roast (brutally honest)... → 977 comments
On runway vs on red carpet ✨😮 #lisa #blackpink #robertwun #f... → 638 comments
An Uncomfortable Met Gala 2026 Red Carpet Fashion Roast... → 626 comments
Everyone came dressed as demons at Met Gala 2026... → 467 comments
BEST DRESSED Met Gala 2026 👏🏻 #metgala #fashionreview... → 542 comments
Met Gala 2026: All the best, worst and weirdest celeb looks... → 421 comments
JENNIE, LISA, JISOO &amp; ROSÉ Serve Iconic Looks at Met Gal... → 450 comments
The Bes